# Clase 7 — Agentes y function calling: el LLM como orquestador

> **¿Cómo pasa un modelo de responder con texto a usar herramientas bajo reglas que definimos?**

En la Clase 6 recuperábamos evidencia con RAG. Hoy vemos otro paso: el modelo propone una acción, nuestro código decide si está permitida, la ejecuta y devuelve el resultado para responder. El ejemplo es un asistente de consultorio que consulta datos ficticios en SQLite.

También vamos a distinguir el agente pequeño que escribimos en Python de un **agent harness**: un entorno reutilizable para coordinar modelos, herramientas, contexto, sesiones y permisos.

---

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Explicar la diferencia entre un modelo LLM, un agente y un agent harness.
- Seguir el ciclo objetivo → propuesta → validación → herramienta → resultado.
- Distinguir un agente a medida en Python de un harness reutilizable.
- Explicar function calling y por qué el modelo propone, pero el programa ejecuta.
- Implementar un dispatch JSON y una herramienta SQL acotada.
- Leer una traza y reconocer qué decisiones se tomaron.
- Separar la licencia del harness de la licencia o modalidad del modelo.
- Elegir entre RAG y fine-tuning según el problema.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Preparar OpenRouter y la base SQLite |
| 2 | Modelo, agente y ciclo de acción |
| 3 | Python a medida y agent harness |
| 4 | Function calling: decidir no es ejecutar |
| 5 | Dispatch JSON manual |
| 6 | La herramienta SQL y sus límites |
| 7 | El agente completo con traza |
| 8 | RAG vs. fine-tuning |
| 9 | Actividad: agregar una herramienta |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en orden.
2. Observá primero el resultado antes de cambiar las variables `TODO`.
3. Ingresá tu clave de OpenRouter en la celda indicada; se guarda en el entorno de esta sesión.
4. Cada consulta de este notebook usa OpenRouter y requiere conexión a Internet.
5. Usá solamente datos ficticios. No pegues historias clínicas ni datos personales reales.

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Modelo LLM | Programa que genera texto o una solicitud estructurada a partir del contexto recibido |
| Agente | Sistema que recibe un objetivo, elige acciones con un modelo y usa herramientas dentro de reglas |
| Agent harness | Entorno de ejecución que prepara contexto, conecta herramientas y controla el ciclo del agente |
| Herramienta | Función o servicio que el agente puede pedir que se ejecute |
| Function calling | Formato para que el modelo solicite una herramienta y sus argumentos |
| Dispatch | Código que valida la solicitud y la dirige a la función correspondiente |
| Estado / sesión | Información que se conserva entre pasos de una misma tarea |
| Permiso | Regla que delimita qué puede leer o cambiar una acción |
| Traza | Registro legible de los pasos y resultados del agente |
| RAG | Recuperar evidencia de una fuente propia antes de generar |
| Fine-tuning | Ajustar los pesos del modelo con ejemplos o datos de entrenamiento |
| Alucinación | Respuesta plausible que puede ser falsa |

## Antes de empezar: cuenta, clave y costos

Como en la Clase 5, usamos OpenRouter. La clave se guarda como variable de entorno.

In [1]:
import json
import ssl
from urllib.request import Request, urlopen

import certifi

OPENROUTER_API_KEY = "api-key"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "openai/gpt-6-luna"
SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())

## 1. Una función para consultar el modelo

Reutilizamos la función de la Clase 5. La diferencia: ahora pedimos una salida **JSON** con `response_format`, porque el agente necesita que el modelo le diga *qué herramienta* usar y *con qué argumentos*, de forma estructurada y no como texto libre.

In [2]:
def consultar_modelo(messages, temperature=0.2, max_tokens=400, response_format=None):
    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "reasoning": {"effort": "none"},
    }
    if response_format is not None:
        payload["response_format"] = response_format

    solicitud = Request(
        OPENROUTER_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
    )
    with urlopen(solicitud, context=SSL_CONTEXT, timeout=60) as respuesta:
        return json.load(respuesta)["choices"][0]["message"]["content"].strip()

## 2. La base SQLite del consultorio

Creamos una base **SQLite embebida** (un archivo local, sin servidor) con dos tablas ficticias: `pacientes` y `turnos`. SQLite viene con Python, así que no hace falta instalar nada.

> **Analogía de la planilla.** Una base relacional es como varias planillas conectadas: una lista de pacientes y otra de turnos, unidas por el número de paciente. SQL es el idioma para preguntarles cosas.

In [3]:
import sqlite3

conexion = sqlite3.connect("consultorio.db")
cursor = conexion.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS turnos;
DROP TABLE IF EXISTS pacientes;

CREATE TABLE pacientes (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    edad INTEGER,
    obra_social TEXT
);

CREATE TABLE turnos (
    id INTEGER PRIMARY KEY,
    paciente_id INTEGER,
    especialidad TEXT,
    fecha TEXT,
    hora TEXT
);
""")

pacientes = [
    (1, "Ana Gómez", 34, "OSDE"),
    (2, "Luis Pérez", 58, "IOMA"),
    (3, "María Ruiz", 41, "PAMI"),
    (4, "Jorge Díaz", 67, "PAMI"),
]
turnos = [
    (1, 1, "Dermatología", "2026-10-12", "09:30"),
    (2, 2, "Cardiología", "2026-10-12", "11:00"),
    (3, 3, "Clínica", "2026-10-13", "10:15"),
    (4, 4, "Cardiología", "2026-10-13", "12:00"),
]

cursor.executemany("INSERT INTO pacientes VALUES (?,?,?,?)", pacientes)
cursor.executemany("INSERT INTO turnos VALUES (?,?,?,?,?)", turnos)
conexion.commit()

print("Base creada. Tablas:", [r[0] for r in cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")])

Base creada. Tablas: ['pacientes', 'turnos']


**Leamos las decisiones.** `executescript` ejecuta varias sentencias a la vez y `DROP TABLE IF EXISTS` nos deja recrear la base cada vez que corremos la celda, sin errores. Los `?` en `executemany` son *placeholders*: SQLite reemplaza cada `?` por un valor de la tupla, lo que evita inyección de SQL (un ataque donde un texto malicioso se mete en la consulta).

## 3. Qué es un agente: del texto a la acción

Un **modelo LLM** recibe texto y genera una continuación. Por sí solo no consulta la base SQLite, no abre archivos y no ejecuta Python: para eso necesita que un programa le conecte esas capacidades.

Para esta clase usamos estas definiciones:

- **Modelo:** propone la próxima respuesta o una solicitud de herramienta.
- **Agente:** sistema que toma un objetivo, usa el modelo para decidir el siguiente paso y puede apoyarse en herramientas.
- **Harness:** el entorno de software que organiza el trabajo: prepara instrucciones y contexto, muestra herramientas disponibles, valida solicitudes, ejecuta acciones, conserva el estado y decide cuándo parar.

El modelo produce una **propuesta**. El código conoce las funciones reales y aplica las reglas antes de ejecutarlas. El resultado vuelve al contexto; después el modelo puede explicarlo con palabras.



**Cómo leerlo:** cada flecha representa información o una acción distinta. La flecha verde que vuelve desde SQL lleva el resultado real; no es una segunda decisión automática del modelo por arte de magia. El ciclo existe porque la aplicación vuelve a llamar al modelo con ese resultado.

> **La autonomía tiene grados.** Depende de qué herramientas se habilitan, cuáles acciones necesitan aprobación y cuántas veces puede repetirse el ciclo. Tener un agente no significa darle permiso para actuar sin límites.

### 3.1 Agente a medida en Python o agent harness

Un agente específico se puede programar a medida en Python. En este notebook ya estamos armando uno: definimos la llamada al modelo, las funciones de SQL y cálculo, el registro de herramientas, las validaciones, el dispatch y la traza. Esas piezas alcanzan para el caso del consultorio y podemos decidir con precisión qué datos y operaciones permitir.

En cambio, un **harness** es una base de ejecución reutilizable. Puede traer un ciclo de varios pasos, manejo de sesiones, conectores, herramientas, permisos, interfaz y registros. En vez de crear cada pieza desde cero, elegimos un modelo, damos instrucciones, configuramos las herramientas y ajustamos los límites disponibles.

La palabra *harness* se usa con distintos alcances según el proyecto. Acá significa el conjunto de código y componentes que permite que el modelo trabaje como agente. Una aplicación propia también tiene un harness, aunque sea pequeño y diseñado para una sola tarea.



| Enfoque | Qué decidimos o construimos | Cuándo puede convenir |
|---|---|---|
| **Python a medida** | Definimos llamadas, funciones, validaciones, datos, ciclo e interfaz que necesita nuestra tarea | El problema es acotado y hace falta control fino sobre las operaciones |
| **Harness listo** | Configuramos las capacidades que ofrece el entorno: modelo, instrucciones, herramientas, memoria, permisos y modo de trabajo | Queremos empezar con capacidades generales o integrarnos a un flujo existente |

Un harness puede ahorrar trabajo de infraestructura, pero **no conoce las reglas de nuestro negocio** hasta que se las explicamos y conectamos. El código propio puede ser pequeño, claro y limitado; también puede requerir más trabajo para implementar sesiones, aprobaciones, reintentos, historial e interfaces.

### 3.2 Código abierto y productos propietarios

**Abierto o cerrado describe el código del programa**, no la inteligencia del modelo. Son decisiones separadas:

- Un harness abierto permite estudiar y modificar el código según su licencia. Igual puede conectarse a un modelo remoto de pago y enviarle solicitudes por Internet.
- Un producto propietario puede permitir instrucciones personalizadas, permisos, extensiones y servidores MCP sin publicar el código de su propio runtime.
- La licencia del programa, la disponibilidad local del modelo, el costo de uso y la protección de datos son preguntas distintas. Hay que revisar cada una.

| Ejemplos | Código del entorno | Qué muestra |
|---|---|---|
| **OpenClaw** y **Hermes Agent** | Sus repositorios se publican con licencia de código abierto | Ofrecen runtimes configurables con herramientas e integraciones; sus funciones dependen de la versión y la configuración |
| **GitHub Copilot** y **Claude Code** | Productos propietarios; el runtime no se publica como un proyecto abierto equivalente | Integran tareas de programación con contexto del repositorio, herramientas y controles configurables |
| **Codex CLI** | El código del CLI está publicado bajo Apache 2.0 | Es la excepción importante: no corresponde clasificar todo lo llamado “Codex” como cerrado |

**Distinción sobre Codex:** el repositorio abierto permite inspeccionar el código del CLI. Eso no permite concluir que cada aplicación o servicio de Codex alojado tenga publicado su código o que el modelo subyacente sea abierto.

Copilot, Claude Code y Codex también ofrecen capacidades que cambian según la aplicación, la suscripción, la configuración y las políticas del equipo. La misma marca puede incluir un editor, una CLI y un servicio alojado con alcances diferentes.

> **Abrir el código no hace seguro a un agente automáticamente.** En un entorno abierto o propietario, revisamos los permisos, las herramientas y las acciones que ejecuta. En el ejemplo de clase, además, la respuesta JSON no basta como control: Python debe validar cada solicitud.

### 3.3 Así puede verse un harness

En esta captura de OpenClaw se ven algunos elementos que el diagrama resume: una interfaz de conversación y un panel para llegar a sesiones y otras capacidades del entorno. La apariencia concreta cambia entre versiones y configuraciones.

![Captura del panel de OpenClaw con chat y controles de sesiones](https://images.contentstack.io/v3/assets/bltefdd0b53724fa2ce/bltffac27a4e3fd8fe9/6a170e5a964ceac34508bc5d/abc051a513b0cc7dff4a7f02493d51e220c72ad4-1999x1095.png)

Imagen de [Elasticsearch Labs: OpenClaw con Elasticsearch](https://www.elastic.co/search-labs/blog/openclaw-elasticsearch-ai-agents).

**Recursos oficiales para comparar capacidades:**

- [OpenClaw: runtime del agente](https://docs.openclaw.ai/agent) y [categorías de herramientas](https://docs.openclaw.ai/tools).
- [Hermes Agent: herramientas y toolsets](https://hermes-agent.nousresearch.com/docs/user-guide/features/tools/) y [repositorio](https://github.com/NousResearch/hermes-agent).
- [GitHub Copilot: usar agentes](https://docs.github.com/en/copilot/how-tos/copilot-in-your-ide/use-copilot-agents).
- [Claude Code: permisos y herramientas de la CLI](https://docs.anthropic.com/en/docs/claude-code/cli-usage).
- [Codex CLI: código y licencia Apache 2.0](https://github.com/openai/codex) (ver el archivo LICENSE).

**Video complementario en español:** [Qué son los agentes de IA y cómo se usan — Aztec Lab](https://www.youtube.com/watch?v=SVf5XEaiP90). Usalo como una introducción general; el notebook muestra un ejemplo concreto en Python.

## 4. Function calling: decidir, no ejecutar

El **function calling** es el mecanismo por el cual el LLM pide ejecutar una herramienta. En lugar de responder con texto libre, el modelo responde con una estructura que dice:

```json
{
  "herramienta": "consultar_sql",
  "argumentos": {"query": "SELECT ..."}
}
```

El modelo **no ejecuta nada**: solo propone. El código valida que la herramienta exista, que los argumentos tengan sentido y recién entonces la ejecuta. Esa validación es la frontera de seguridad.

## 5. Dispatch JSON manual

Vamos a implementar el dispatch a mano, sin librerías de agentes, para ver el mecanismo con claridad. Primero definimos un **registro de herramientas**: un diccionario que describe cada herramienta disponible y la función que la ejecuta.

In [4]:
def consultar_sql(query):
    """Ejecuta una consulta SQL de SOLO LECTURA y devuelve las filas."""
    consulta = query.strip().lower()
    if not consulta.startswith("select"):
        return {"error": "Solo se permiten consultas SELECT (lectura)."}
    try:
        cursor.execute(query)
        columnas = [d[0] for d in cursor.description]
        filas = cursor.fetchall()
        return {"columnas": columnas, "filas": filas}
    except Exception as error:
        return {"error": str(error)}


def calcular_edad(anio_nacimiento):
    """Calcula la edad aproximada a partir del año de nacimiento."""
    return {"edad": 2026 - int(anio_nacimiento)}


REGISTRO_HERRAMIENTAS = {
    "consultar_sql": {
        "descripcion": "Consulta la base del consultorio (pacientes y turnos). Solo lectura.",
        "funcion": consultar_sql,
    },
    "calcular_edad": {
        "descripcion": "Calcula la edad a partir del año de nacimiento.",
        "funcion": calcular_edad,
    },
}

print("Herramientas registradas:", list(REGISTRO_HERRAMIENTAS))

Herramientas registradas: ['consultar_sql', 'calcular_edad']


**Leamos las decisiones.** La whitelist de solo `SELECT` es la protección clave: aunque el LLM proponga un `DELETE` o un `DROP`, la función lo rechaza antes de tocar la base. El `try/except` convierte cualquier error de SQL en un mensaje legible en vez de romper el notebook.

## 6. El dispatch: del JSON a la ejecución

Ahora la pieza central: una función que recibe la decisión del LLM (JSON), la valida contra el registro y ejecuta la herramienta correcta.

In [5]:
def ejecutar_decision(decision):
    """Recibe la decisión del LLM y ejecuta la herramienta indicada."""
    nombre = decision.get("herramienta")
    argumentos = decision.get("argumentos", {})

    if nombre not in REGISTRO_HERRAMIENTAS:
        return {"error": f"Herramienta desconocida: {nombre}"}

    herramienta = REGISTRO_HERRAMIENTAS[nombre]["funcion"]
    try:
        resultado = herramienta(**argumentos)
    except TypeError as error:
        return {"error": f"Argumentos inválidos para {nombre}: {error}"}
    return {"herramienta": nombre, "resultado": resultado}


# Probamos el dispatch con decisiones manuales (sin LLM todavía)
for decision in [
    {"herramienta": "consultar_sql", "argumentos": {"query": "SELECT nombre, edad FROM pacientes"}},
    {"herramienta": "consultar_sql", "argumentos": {"query": "DELETE FROM pacientes"}},
    {"herramienta": "calcular_edad", "argumentos": {"anio_nacimiento": 1990}},
    {"herramienta": "no_existe", "argumentos": {}},
]:
    print(ejecutar_decision(decision))

{'herramienta': 'consultar_sql', 'resultado': {'columnas': ['nombre', 'edad'], 'filas': [('Ana Gómez', 34), ('Luis Pérez', 58), ('María Ruiz', 41), ('Jorge Díaz', 67)]}}
{'herramienta': 'consultar_sql', 'resultado': {'error': 'Solo se permiten consultas SELECT (lectura).'}}
{'herramienta': 'calcular_edad', 'resultado': {'edad': 36}}
{'error': 'Herramienta desconocida: no_existe'}


**Leamos las decisiones.** El dispatch hace tres validaciones: que la herramienta exista, que los argumentos sean compatibles (`TypeError`) y que la consulta sea de solo lectura (dentro de `consultar_sql`). Noten que el `DELETE` fue rechazado: el LLM podría proponerlo, pero la herramienta no lo ejecuta.

## 7. El agente completo: el LLM decide

Ahora unimos todo. El agente:

1. Le pasa al LLM la pregunta del usuario **y la lista de herramientas disponibles** (sus descripciones).
2. El LLM responde con JSON: qué herramienta y con qué argumentos.
3. El dispatch ejecuta la herramienta.
4. El resultado vuelve al LLM, que redacta la respuesta final.

Guardamos una **traza** de cada paso para poder auditar qué hizo el agente.

In [6]:
def describir_herramientas():
    return "\n".join(
        f"- {nombre}: {info['descripcion']}"
        for nombre, info in REGISTRO_HERRAMIENTAS.items()
    )


def agente(pregunta):
    traza = []

    # Paso 1: el LLM decide qué herramienta usar
    mensajes = [
        {"role": "system", "content": (
            "Sos el asistente de un consultorio. Para responder, elegí UNA "
            "herramienta de la lista y devolvé un JSON con los campos "
            "'herramienta' y 'argumentos'. Si la pregunta no requiere "
            "herramienta, devolvé 'herramienta': 'ninguna'. "
            f"Herramientas disponibles:\n{describir_herramientas()}"
        )},
        {"role": "user", "content": pregunta},
    ]
    salida = consultar_modelo(
        mensajes,
        response_format={"type": "json_object"},
    )
    decision = json.loads(salida)
    traza.append({"paso": "decidir", "decision": decision})

    # Paso 2: si no hay herramienta, respondemos directo
    if decision.get("herramienta") == "ninguna":
        return {"respuesta": "No necesito consultar datos para eso.", "traza": traza}

    # Paso 3: ejecutar la herramienta
    ejecucion = ejecutar_decision(decision)
    traza.append({"paso": "ejecutar", "ejecucion": ejecucion})

    # Paso 4: el LLM redacta la respuesta con el resultado real
    mensajes.append({"role": "assistant", "content": salida})
    mensajes.append({
        "role": "user",
        "content": f"Resultado de la herramienta: {json.dumps(ejecucion, ensure_ascii=False)}. Redactá una respuesta clara al usuario.",
    })
    respuesta = consultar_modelo(mensajes, max_tokens=250)
    traza.append({"paso": "redactar"})

    return {"respuesta": respuesta, "traza": traza}

Probemos el agente con una pregunta que requiere consultar la base.

In [ ]:
resultado = agente("¿Qué turnos de cardiología hay la semana que viene?")
print(resultado["respuesta"])
print("\n--- Traza ---")
for paso in resultado["traza"]:
    print(paso)

**Para interpretar.** Mirá la traza: el agente decidió usar `consultar_sql`, ejecutó la consulta y redactó la respuesta con el resultado real.

**Límite de este ejemplo:** cada pregunta genera como máximo una llamada a una herramienta. No repetimos un ciclo `while` en el que el modelo pueda pedir herramientas adicionales hasta terminar. Es una versión mínima para ver cada pieza por separado. Un harness de programación puede permitir varios pasos, pero también necesita límites de iteraciones, permisos y revisión humana según la tarea.

## 8. RAG vs Fine-tuning: criterios de decisión

En la Clase 6 vimos RAG (recuperar evidencia de una fuente propia). El fine-tuning es otra forma de adaptar un modelo. ¿Cuándo usar cada uno?

| Criterio | RAG | Fine-tuning |
|---|---|---|
| ¿Qué cambia? | Los datos (documentos, base) | El comportamiento del modelo |
| ¿Qué se entrena? | Nada: se busca y se inyecta | Los pesos del modelo |
| ¿Cuándo actualizar? | Editando la fuente | Reentrenando con datos nuevos |
| Costo | Bajo (solo búsqueda) | Alto (GPU, datos, tiempo) |
| Transparencia | Alta: se ve la fuente | Baja: caja negra |
| Mejor para | Datos propios que cambian | Tono, estilo, formato de salida |

```text
¿El conocimiento es tuyo y cambia?  ->  RAG
¿El problema es de estilo/formato?  ->  Fine-tuning
¿Necesitás ambos?                  ->  Combinarlos
```

> **Regla práctica:** si el problema es *saber algo* (hechos, datos, reglas), RAG. Si el problema es *comportarse de cierta forma* (tono, formato, tarea específica), fine-tuning. Para datos propios que cambian, RAG casi siempre gana por costo y actualización.

---
# Parte 2 — Actividad: agregar una herramienta (25 minutos)

## La consigna

El consultorio quiere que el asistente también responda preguntas sobre **obra social** de los pacientes. Tu tarea: agregar una herramienta nueva al registro y probar que el agente la usa.

### Tarea 1 — Agregar la herramienta (10 minutos)

Creá una función `consultar_obra_social(nombre)` que consulte la base y devuelva la obra social del paciente. Registrala en `REGISTRO_HERRAMIENTAS` con una descripción clara.

In [ ]:
# TODO: definí la función y registrala
def consultar_obra_social(nombre):
    # Completá: SELECT obra_social FROM pacientes WHERE nombre = ?
    pass

# TODO: agregá la herramienta al REGISTRO_HERRAMIENTAS
# REGISTRO_HERRAMIENTAS["consultar_obra_social"] = {...}

### Tarea 2 — Probar el agente (10 minutos)

Ejecutá el agente con una pregunta que requiera la herramienta nueva.

In [ ]:
# TODO: probá el agente con la herramienta nueva
# resultado = agente("¿Qué obra social tiene Ana Gómez?")
# print(resultado["respuesta"])
# print(resultado["traza"])

### Tarea 3 — Medir y reflexionar (5 minutos)

Respondé brevemente:

1. ¿Qué pasó en la traza cuando el agente usó la herramienta nueva?
2. ¿Qué pasa si el LLM propone una consulta que borra datos? ¿Dónde se detiene?
3. ¿Qué riesgo corrés si le das al LLM una herramienta que *escribe* en la base?
4. ¿Este problema (obra social) se resuelve mejor con RAG o con la herramienta SQL? ¿Por qué?

---
## ✅ Síntesis de la clase

1. El **modelo** propone texto o una acción estructurada; el **agente** organiza decisiones y acciones para cumplir un objetivo.
2. El **harness** prepara el contexto, conecta herramientas, controla los pasos y registra resultados.
3. En un agente a medida escribimos las piezas necesarias en Python; un harness reutilizable trae partes generales listas para configurar.
4. **Function calling** expresa una solicitud de herramienta; no la ejecuta por sí mismo.
5. El código valida la propuesta antes de llamar a una herramienta y la traza permite revisar qué ocurrió.
6. Código abierto o propietario describe el harness. El modelo, su licencia, dónde corre y cuánto cuesta son decisiones aparte.
7. **RAG** aporta evidencia recuperada; **fine-tuning** ajusta el comportamiento; pueden combinarse.

> Un agente no obtiene autoridad por escribir una solicitud. La autoridad viene de las herramientas y los permisos que el programa le habilita.